# Advanced feature engineering and ablation

Run from top to bottom. Quick mode is a patient subset, not final results. Notebook outputs are separate from production reports.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'features.py').exists()), None)
assert ROOT is not None, 'Open the notebook from the project repository'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.notebook_utils import *
QUICK = True  # False uses the entire eligible dataset; keep consistent across notebooks
OUT = ROOT/'reports'/('notebook_quick' if QUICK else 'notebook_full')
OUT.mkdir(parents=True, exist_ok=True)


Load cohort

In [ ]:
raw, x, y, g, parts = load_cohort(ROOT, QUICK)
train, cal, tune, test = (parts[k] for k in ("train","calibration","threshold","test"))
X, Y, G = x.iloc[train], y.iloc[train], g.iloc[train]
display(split_table(y,g,parts))

Construct eleven additional features and define feature groups.

In [ ]:
from src.advanced_experiment import expanded,pipeline
z,groups=expanded(raw); A=z.iloc[train]; extra=sum(groups.values(),[]); base=[c for c in A if c not in extra]
configs={'baseline':base,'all_features':list(A)}
for k,v in groups.items(): configs['add_'+k]=base+v; configs['remove_'+k]=[c for c in A if c not in v]
display(A[extra].head()); print(groups)

Perform grouped cross-validation with identical folds

In [ ]:
rows=[]
for name,cols in configs.items():
    _,r=group_cv(pipeline(A[cols],QUICK),A[cols],Y,G)
    for i,(ap,auc) in enumerate(r): rows.append({'config':name,'fold':i,'ap':ap,'roc_auc':auc})
d=pd.DataFrame(rows); summary=d.groupby('config').ap.agg(['mean','std']).sort_values('mean',ascending=False); display(summary); summary['mean'].plot.barh(xerr=summary['std']); plt.tight_layout(); plt.show(); d.to_csv(OUT/'ablation_folds.csv',index=False)